# 01 — Bronze Layer: Raw Data Ingestion

This notebook demonstrates the **Bronze layer** of the B3 Data Platform.

**Goal:** ingest raw daily OHLCV prices from Yahoo Finance and persist them as-is
(no transformation) into the Bronze Parquet store.

Architecture reminder:
```
Yahoo Finance API  ──►  Bronze (raw Parquet)  ──►  Silver  ──►  Gold
```
**Rules for Bronze:**
- Data is written exactly as received — no value changes.
- Only metadata columns are added: `source` and `ingested_at`.
- Partitioned by `trade_date` for fast downstream reads.

In [ ]:
# Configuração básica do ambiente do notebook.
# - `sys.path.insert(0, "..")` permite importar os módulos do projeto
#   (a_configs, c_ingestion, d_processing, ...) a partir do diretório raiz.
# - Polars é a engine principal de manipulação de dados (camadas Bronze/Silver).
import sys
sys.path.insert(0, "..")

from datetime import date, timedelta

import polars as pl

# Exibe até 20 linhas por DataFrame para facilitar a inspeção visual.
pl.Config.set_tbl_rows(20)
print(f"Polars version: {pl.__version__}")

## 1. Fetch raw prices from Yahoo Finance

In [ ]:
# 1) Extração: baixa os preços diários (OHLCV) direto do Yahoo Finance.
#    Este é o ÚNICO ponto onde o notebook fala com a fonte externa.
from c_ingestion.yahoo_finance import fetch_daily_prices

# Janela de 3 meses encerrando hoje — suficiente para demonstração.
END   = date.today()
START = END - timedelta(days=90)

# Subconjunto reduzido (5 ativos) usado apenas no exemplo interativo.
# O pipeline completo (célula final) usa a lista DEFAULT_TICKERS de a_configs.
DEMO_TICKERS = ["PETR4.SA", "VALE3.SA", "ITUB4.SA", "BBDC4.SA", "ABEV3.SA"]

raw_df = fetch_daily_prices(tickers=DEMO_TICKERS, start=START, end=END)
print(f"Fetched {len(raw_df):,} rows for {raw_df['ticker'].n_unique()} tickers")
raw_df.head(10)

## 2. Inspect raw data

In [ ]:
# 2a) Inspeção rápida: schema, contagem de nulos e estatística descritiva.
# Útil para validar que o download retornou dados consistentes antes de gravar.
print("Schema:")
print(raw_df.schema)
print("\nNull counts:")
print(raw_df.null_count())
print("\nBasic stats:")
raw_df.describe()

In [ ]:
# 2b) Cobertura temporal por ativo.
# Conferimos primeira/última data e quantidade de pregões — qualquer ativo
# com poucos pregões pode indicar problema no download ou ticker suspenso.
raw_df.group_by("ticker").agg(
    pl.col("trade_date").min().alias("first_date"),
    pl.col("trade_date").max().alias("last_date"),
    pl.col("trade_date").n_unique().alias("trading_days"),
).sort("ticker")

## 3. Write to Bronze layer

In [ ]:
# 3) Carga na camada Bronze.
# Grava os dados como Parquet particionado por trade_date.
# A camada Bronze é IMUTÁVEL: não alteramos valores, apenas adicionamos
# colunas de metadata (`source` e `ingested_at`) para rastreabilidade.
from d_processing.bronze.ingest import write_bronze

bronze_path = write_bronze(raw_df, source="yahoo_finance")
print(f"Written to: {bronze_path}")

## 4. Read back from Bronze and verify

In [ ]:
# 4) Releitura: garante que o write foi consistente e mostra as colunas
# de metadata acrescentadas pela camada Bronze.
from d_processing.bronze.ingest import read_bronze

bronze_df = read_bronze(source="yahoo_finance")
print(f"Bronze rows: {len(bronze_df):,}")
print(f"Columns added: {[c for c in bronze_df.columns if c not in raw_df.columns]}")
bronze_df.head(10)

## 5. Quick visualization — closing prices

In [ ]:
# 5) Visualização: preços de fechamento por ativo ao longo do tempo.
# Usamos Plotly Express para ter um gráfico interativo (zoom, hover, legenda).
# Template `plotly_white` melhora a leitura em apresentações/projeção.
import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_white"

# Plotly recebe pandas, por isso o `.to_pandas()` no final do encadeamento.
fig = px.line(
    bronze_df.sort("trade_date").to_pandas(),
    x="trade_date",
    y="close",
    color="ticker",
    title="Preços de Fechamento das Ações da B3 — Camada Bronze (dados brutos)",
    labels={
        "trade_date": "Data do Pregão",
        "close": "Preço de Fechamento (R$)",
        "ticker": "Ativo",
    },
)
# Ajustes de layout pensados em legibilidade para apresentação.
fig.update_layout(
    height=500,
    hovermode="x unified",                       # tooltip único para todos os ativos
    legend=dict(title="Ativo", orientation="v", x=1.02, y=1),
    margin=dict(l=60, r=40, t=70, b=50),
)
fig.update_yaxes(tickprefix="R$ ", separatethousands=True)
fig.update_xaxes(tickformat="%d/%m/%Y")
fig.show()

## 6. Run full Bronze pipeline (all default tickers)

In [ ]:
# 6) Execução do pipeline completo da camada Bronze.
# Mesmo fluxo extract→load das células acima, mas encapsulado numa classe
# (BronzePipeline) que é a unidade chamada pelas DAGs do Airflow.
from f_pipelines.bronze_pipeline import BronzePipeline, BronzePipelineConfig

cfg = BronzePipelineConfig(start=START, end=END)
pipeline = BronzePipeline(config=cfg)
result = pipeline.run()
print(f"Pipeline produced {len(result):,} rows")